In [1]:
%pip install numpy-financial
import numpy_financial as npf
print(npf.__version__)


[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


1.0.0


In [2]:
# Cell 1: load your existing engine (CRAH/WSE/Chiller/Tower)
%run "./Clean CRAH Financial Comparison Model.ipynb"   


Loaded source — Phoenix: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx
Loaded source — Fairbanks: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx
Loaded rows — Phoenix: 8760 Fairbanks: 8760
Qserver first hour — Phoenix: 1342.595476084541 Fairbanks: 1342.595476084541

=== PHOENIX — Hourly (first 6 of 8760) ===
                 mode        IT_kW  Twb_out_C  P_sys_mag_kW  P_sys_cen_kW  Δ_kW (cen - mag)
0       MODE3_CHILLER  1342.595476      11.18    100.594686    114.760547         14.165861
1       MODE3_CHILLER  1342.685053       9.79     95.943290    107.190236         11.246946
2       MODE3_CHILLER  1340.395705       8.49     91.698479    100.542454          8.843975
3  MODE2_PARTIAL_FREE  1338.474414       7.79     85.843891     93.438577          7.594686
4       MODE3_CHILLER  1339.993680       8.03     90.290431     98.355683          8.065253
5  MODE2_PARTIAL_FREE  1335.405900       6.58  

In [3]:
# # =============================================================================
# # BLOCK 1 — INPUTS
# # Assumes your engine (PlantParams, plan_power_timeseries) is already loaded via:
# #   %run "./Thesis Financial Model CRAH.ipynb"
# # =============================================================================

# import numpy as np
# import pandas as pd
# from dataclasses import dataclass

# # ─────────────────────────────────────────────────────────────────────────────
# # SITES & TARIFFS  (HARD-CODED per your instruction)
# # ─────────────────────────────────────────────────────────────────────────────
# SITES = {
#     "Fairbanks_Alaska": {"energy_price_usd_per_kWh": 0.25},  # flat price; no TOU/demand for now
#     "Phoenix_Arizona":  {"energy_price_usd_per_kWh": 0.15},
# }

# # ─────────────────────────────────────────────────────────────────────────────
# # FINANCE KNOBS  (HARD-CODED)
# # Real discount & escalation (used in NPV/IRR/payback)
# # Ref style mirrors LBNL/RTES finance sections (20-yr life commonly used).
# # ─────────────────────────────────────────────────────────────────────────────
# DISCOUNT_RATE = 0.05   # 5% real discount
# ESCALATION    = 0.02   # 2% real escalation of energy costs/savings
# LIFE_YEARS    = 20

# # ─────────────────────────────────────────────────────────────────────────────
# # BASELINE (CRAH/WSE/Chiller) CAPEX ASSUMPTIONS
# # ─────────────────────────────────────────────────────────────────────────────
# # Chiller CAPEX — simple $/ton rule (LBNL-style correlation; keep equal unless you have vendor deltas)
# CHILLER_CAPEX_PER_TON_USD_CEN = 800.0
# CHILLER_CAPEX_PER_TON_USD_MAG = 900.0  # HARD-CODED

# # Common piping allowance — applied to BOTH baseline and ATES (apples-to-apples)
# # If you have RTES table lengths and $/m, paste them here later.
# PIPING_TOTAL_LENGTH_M = 500.0       # ASSUMED total supply+return length (m)
# PIPING_COST_PER_M_USD = 35.0      # ASSUMED installed $/m #Based out of simple google search

# # “Other” baseline costs (controls/commissioning/engineering) — parity placeholder
# BASELINE_FIXED_INSTALL_USD = 150_000.0  # ASSUMED
# BASELINE_OPEX_PCT_PER_YEAR = 0.05       # ASSUMED (5% of CapEx/yr) — same across plans for parity

# # ─────────────────────────────────────────────────────────────────────────────
# # ATES CONFIG (per site): temps, wells, drilling, hydraulics, HX, pumps
# # Sources used for defaults in comments:
# # • Ford Site ATES (2016) — geometry/flows/efficiency style (good proxy for Fairbanks_Alaska)
# # • Your “drilling well cost” note — $300/m (you provided)
# # • Your prior choices — Phoenix_Arizona cold/hot 6/15 °C; total flow 1000 m³/h
# # ─────────────────────────────────────────────────────────────────────────────

# @dataclass
# class ATESConfig:
#     # Thermal targets (plant side). We gate discharge on: cold_well_C + hx_approach_K ≤ plant T_supply_set (6 °C).
#     cold_well_C: float
#     hot_well_C:  float
#     hx_approach_K: float = 2.0    # ASSUMED approach across plate HX (tight plates ~1–2 K typical)

#     # Storage (very simple bucket model for now)
#     storage_cap_kWh_th: float = 1_800_000.0  # ASSUMED seasonable capacity (you can replace with ρ·cp·V_pore·ΔT)
#     standing_loss_pct_per_month: float = 0.03  # ASSUMED monthly standing loss; swap to UA·ΔT later if you wish

#     # Wells & drilling
#     n_cold_wells: int = 5      # YOUR choice (five cold wells)
#     n_warm_wells: int = 5      # YOUR choice (five warm wells)
#     well_depth_m: float = 80.0 # YOUR constraint: 70–80 m; we use 80 m
#     drilling_cost_per_m_usd: float = 300.0  # YOUR input: $300/m
#     per_well_fixed_adder_usd: float = 1892.0  # YOUR note: fixed adder per well
#     casing_screen_adder_usd: float = 0.0      # ASSUMED (set if you have it)

#     # Hydraulics / flow (plant design choice)
#     total_flow_m3_per_h: float = 1000.0  # YOUR request
#     head_m: float = 25.0                 # ASSUMED total dynamic head (static + friction); refine later
#     pump_hydraulic_eff: float = 0.75     # ASSUMED
#     motor_eff: float = 0.95              # ASSUMED
#     vfd_eff: float = 0.98                # ASSUMED

#     # Pump cost rule (CapEx)
#     pump_cost_per_kW_usd: float = 350.0  # YOUR note: $350/kW; 300–400 kW class

#     # ATES plate HX CapEx — simple linear proxy (adjust when you have vendor quote)
#     hx_capex_a_usd_per_kW: float = 50.0  # ASSUMED slope
#     hx_capex_b_usd: float = 20_000.0     # ASSUMED intercept

#     # Dispatch power limits (thermal)
#     qmax_discharge_kW: float = 4000.0    # ASSUMED (cap on ATES direct cooling)
#     qmax_charge_kW: float = 3000.0       # ASSUMED
#     charge_when_twb_below_C: float = 10.0  # ASSUMED: charge when outdoor WB ≤ 10 °C

# # Site-specific ATES configs
# ATES_BY_SITE = {
#     # Phoenix_Arizona — YOU specified 6/15 °C. Keep Ford-style geometry out for now (don’t change everything yet).
#     "Phoenix_Arizona":  ATESConfig(cold_well_C=8.0, hot_well_C=18.0),
#     # Fairbanks_Alaska — we keep similar defaults but (if you want later) can move toward Ford values (cooler aquifer).
#     "Fairbanks_Alaska": ATESConfig(cold_well_C=5.0, hot_well_C=12.0),
# }
# print("Executed")

In [4]:
# # =============================================================================
# # BLOCK 2 — CLASSES & CALCULATIONS
# #  - ATESState (SOC bucket)
# #  - Pump power, CAPEX/OPEX helpers
# #  - NPV/IRR helpers (no external packages)
# #  - plan_power_timeseries_with_ates: inserts ATES BEFORE chiller with safety guards
# # =============================================================================

# class ATESState:
#     """Simple energy bucket for storage (kWh_th) with per-hour standing loss."""
#     def __init__(self, init_kWh_th=None, cap_kWh_th=150_000.0, monthly_loss_pct=0.03):
#         self.cap_kWh_th = float(cap_kWh_th)
#         self.E_kWh_th = float(init_kWh_th) if init_kWh_th is not None else 0.5 * self.cap_kWh_th
#         # Convert monthly % loss to hourly factor (≈ 30-day month)
#         self.loss_per_hour = 1.0 - (1.0 - float(monthly_loss_pct))**(1.0/(30.0*24.0))

# def clamp_nonneg(x: float) -> float:
#     """Ensure no negative values propagate."""
#     return float(x) if x > 0.0 else 0.0

# def pump_power_kW(flow_m3h: float, head_m: float, eta_h: float, eta_motor: float, eta_vfd: float) -> float:
#     """
#     Hydraulic pump power (kW) based on flow & head:
#       P_hydraulic = ρ g Q * head   (Q in m³/s), then divide by total η.
#     Guards ensure non-negatives.
#     """
#     rho = 1000.0   # kg/m³
#     g   = 9.81     # m/s²
#     Q_m3s = max(0.0, flow_m3h) / 3600.0
#     head = max(0.0, head_m)
#     hydraulic_kW = rho * g * Q_m3s * head / 1000.0
#     eta_total = max(1e-3, (eta_h or 0.0) * (eta_motor or 0.0) * (eta_vfd or 0.0))
#     return hydraulic_kW / eta_total

# def ates_wellfield_capex_usd(cfg: ATESConfig) -> float:
#     """Drilling + fixed per-well adders (no pumps/HX/piping here)."""
#     wells_total = int(max(0, cfg.n_cold_wells) + max(0, cfg.n_warm_wells))
#     drilling = max(0.0, cfg.well_depth_m) * max(0.0, cfg.drilling_cost_per_m_usd) * wells_total
#     fixed = wells_total * (max(0.0, cfg.per_well_fixed_adder_usd) + max(0.0, cfg.casing_screen_adder_usd))
#     return drilling + fixed

# def ates_pump_capex_usd(cfg: ATESConfig) -> float:
#     """Size well pumps from total flow/head and apply $/kW rule (your $350/kW)."""
#     P_kW = pump_power_kW(cfg.total_flow_m3_per_h, cfg.head_m,
#                          cfg.pump_hydraulic_eff, cfg.motor_eff, cfg.vfd_eff)
#     return max(0.0, P_kW) * max(0.0, cfg.pump_cost_per_kW_usd)

# def ates_hx_capex_usd(duty_kW: float, cfg: ATESConfig) -> float:
#     """Plate HX proxy: CapEx = a * duty + b (positive only)."""
#     a, b = max(0.0, cfg.hx_capex_a_usd_per_kW), max(0.0, cfg.hx_capex_b_usd)
#     return a * max(0.0, duty_kW) + b

# def piping_capex_usd() -> float:
#     """Common piping allowance used for BOTH Baseline and ATES (apples-to-apples)."""
#     return max(0.0, PIPING_TOTAL_LENGTH_M) * max(0.0, PIPING_COST_PER_M_USD)

# def chiller_capex_usd_from_peak_Q(peak_Q_kW: float, per_ton_usd: float = CHILLER_CAPEX_PER_TON_USD_MAG) -> float:
#     """1 refrigeration ton = 3.517 kW of cooling."""
#     tons = max(0.0, peak_Q_kW) / 3.517
#     return tons * max(0.0, per_ton_usd)

# def npv_of_savings_usd(annual_savings_usd: float, years=LIFE_YEARS, d=DISCOUNT_RATE, esc=ESCALATION) -> float:
#     """Growing annuity NPV; guards to avoid NaN/neg blowups."""
#     S = max(0.0, float(annual_savings_usd))
#     y = int(max(0, years))
#     d = float(d); esc = float(esc)
#     total = 0.0
#     for t in range(1, y+1):
#         total += S * ((1+esc)**(t-1)) / ((1+d)**t)
#     return total

# def irr_on_premium(premium_usd: float, annual_savings_usd: float, years=LIFE_YEARS, esc=ESCALATION) -> float:
#     """IRR via bisection on cashflow: [-premium, +S(1+esc)^1, ..., +S(1+esc)^N]."""
#     prem = float(premium_usd)
#     S    = float(annual_savings_usd)
#     y    = int(max(1, years))
#     if prem <= 0 and S <= 0:
#         return float('nan')
#     cash = [-prem] + [S * ((1+esc)**t) for t in range(1, y+1)]
#     lo, hi = -0.9, 1.5
#     for _ in range(200):
#         mid = 0.5*(lo+hi)
#         npv = sum(cash[t] / ((1+mid)**t) for t in range(len(cash)))
#         if npv > 0: lo = mid
#         else:       hi = mid
#     return 0.5*(lo+hi)

# def kpis_from_results(res_df: pd.DataFrame):
#     """Compute annual kWh, peak kW, and PUE′ stats from your hourly results. Guards to avoid negatives."""
#     it_kW = np.maximum(1e-9, res_df['Qsum_kW'].to_numpy(dtype=float) / 1.10)  # IT = Qsum / 1.10
#     Psys  = np.maximum(0.0, res_df['P_sys_kW'].to_numpy(dtype=float))
#     kWh   = float(Psys.sum())        # 1 row = 1 hour
#     peak  = float(Psys.max())
#     puep  = 1.0 + (Psys / it_kW)
#     return {
#         "kWh": kWh,
#         "peak_kW": peak,
#         "PUEp_avg": float(np.mean(puep)),
#         "PUEp_p95": float(np.quantile(puep, 0.95)),
#     }

# def plan_power_timeseries_with_ates(df_hourly: pd.DataFrame,
#                                     plant: PlantParams,
#                                     ates_cfg: ATESConfig,
#                                     state: ATESState,
#                                     dt_hours: float = 1.0) -> pd.DataFrame:
#     """
#     Reuse your existing plant model to compute WSE & baseline splits,
#     then insert ATES BEFORE chiller:
#       • Feasibility: (cold_well_C + hx_approach_K) ≤ plant.Tcw_supply_set_C
#       • Discharge limited by (need, qmax_discharge, SOC/dt)
#       • Optional charging during cool WB hours
#       • Update chiller power and total P_sys with ATES pump power
#     Safety guards ensure no negative Q/P/SOC.
#     """
#     base = plan_power_timeseries(df_hourly.copy(), plant)

#     can_supply = (ates_cfg.cold_well_C + ates_cfg.hx_approach_K) <= plant.Tcw_supply_set_C
#     P_ates_pump_kW = pump_power_kW(ates_cfg.total_flow_m3_per_h, ates_cfg.head_m,
#                                    ates_cfg.pump_hydraulic_eff, ates_cfg.motor_eff, ates_cfg.vfd_eff)

#     base['Q_ates_kW'] = 0.0
#     base['P_ates_pumps_kW'] = 0.0
#     base['mode_with_ates'] = base['mode'].values

#     for i in range(len(base)):
#         Qsum  = max(0.0, float(base.at[i, 'Qsum_kW']))
#         Qwse  = max(0.0, float(base.at[i, 'Q_wse_kW']))
#         need  = max(0.0, Qsum - Qwse)

#         # ── ATES DISCHARGE ────────────────────────────────────────────────────
#         Q_ates = 0.0
#         if need > 1e-9 and can_supply and state.E_kWh_th > 1e-9:
#             E_avail_kW = max(0.0, state.E_kWh_th / max(1e-9, dt_hours))
#             Q_ates = min(need, ates_cfg.qmax_discharge_kW, E_avail_kW)
#             Q_ates = max(0.0, Q_ates)
#             state.E_kWh_th = max(0.0, state.E_kWh_th - Q_ates * dt_hours)
#             base.at[i, 'Q_ates_kW'] = Q_ates
#             base.at[i, 'P_ates_pumps_kW'] = P_ates_pump_kW
#             base.at[i, 'mode_with_ates'] = ("MODE1_FULL_FREE" if abs(Q_ates - need) < 1e-6
#                                             else "MODE2_PARTIAL_FREE_ATES")

#         # ── OPTIONAL CHARGING (cool WB hours) ────────────────────────────────
#         Twb = float(df_hourly.iloc[i]['Twb_out_C'])
#         if (state.E_kWh_th < state.cap_kWh_th) and (Twb <= ates_cfg.charge_when_twb_below_C):
#             Q_charge = min(ates_cfg.qmax_charge_kW,
#                            (state.cap_kWh_th - state.E_kWh_th)/max(1e-9, dt_hours))
#             if Q_charge > 1e-9:
#                 base.at[i, 'P_ates_pumps_kW'] += P_ates_pump_kW  # charge pumping power
#                 state.E_kWh_th = min(state.cap_kWh_th, state.E_kWh_th + Q_charge * dt_hours)

#         # ── STANDING LOSS at end of hour ─────────────────────────────────────
#         state.E_kWh_th = max(0.0, state.E_kWh_th * (1.0 - state.loss_per_hour))

#         # ── UPDATE CHILLER & SYSTEM POWER ────────────────────────────────────
#         Q_ch_old = max(0.0, float(base.at[i, 'Q_chiller_kW']))
#         Q_ch_new = max(0.0, Q_ch_old - Q_ates)
#         base.at[i, 'Q_chiller_kW'] = Q_ch_new

#         COP = float(base.at[i, 'COP']) if (base.at[i, 'COP'] == base.at[i, 'COP']) else 0.0
#         P_ch_new = (Q_ch_new / max(1e-9, COP)) if Q_ch_new > 0.0 and COP > 0.0 else 0.0
#         base.at[i, 'P_chiller_kW'] = P_ch_new

#         # Sum all components + ATES pumps; guard against negatives
#         P_sys = (P_ch_new
#                  + max(0.0, float(base.at[i, 'P_pumps_chw_kW']))
#                  + max(0.0, float(base.at[i, 'P_pumps_ctw_kW']))
#                  + max(0.0, float(base.at[i, 'P_tower_fans_kW']))
#                  + max(0.0, float(base.at[i, 'P_crah_kW']))
#                  + max(0.0, float(base.at[i, 'P_ates_pumps_kW'])))
#         base.at[i, 'P_sys_kW'] = max(0.0, P_sys)

#     return base
# print("Executed")

In [5]:
# # =============================================================================
# # BLOCK 3 — OUTPUTS
# #  Plans: Centrifugal (baseline), Magnetic, ATES+Magnetic
# #  KPIs:  kWh, $/yr, peak kW, PUE′;  Finance: CapEx, ΔkWh/$, NPV, IRR, payback
# # =============================================================================
# # ---- Site key resolver (put this above run_site_compare) ----
# SITE_KEY_ALIASES = {
#     "Fairbanks_Alaska": "Fairbanks",
#     "Phoenix_Arizona":  "Phoenix",
#     "Fairbanks, AK":    "Fairbanks",
#     "Phoenix, AZ":      "Phoenix",
# }

# def _resolve_site_key(site_name: str, mapping: dict, sites_dict: dict) -> str:
#     # 1) exact alias
#     key = mapping.get(site_name, site_name)
#     if key in sites_dict:
#         return key
#     # 2) fallback: case-insensitive containment match
#     lower = site_name.lower()
#     for k in sites_dict.keys():
#         if k.lower() in lower or lower in k.lower():
#             return k
#     # 3) clear error with available keys
#     raise KeyError(f"Site '{site_name}' not found. Available: {list(sites_dict.keys())}")


# def run_site_compare(df_hourly: pd.DataFrame, site_name: str):
#     key      = _resolve_site_key(site_name, SITE_KEY_ALIASES, SITES)
#     ates_key = _resolve_site_key(site_name, SITE_KEY_ALIASES, ATES_BY_SITE)
#     price    = float(SITES[key]["energy_price_usd_per_kWh"])
#     ates_cfg = ATES_BY_SITE[ates_key]

#     # Build plant params (from your engine)
#     p_cen = PlantParams(chiller_type="centrifugal", chiller_cap_kW_per_unit=4058.0)
#     p_mag = PlantParams(chiller_type="magnetic",    chiller_cap_kW_per_unit=3900.0)

#     # Physics runs
#     res_cen = plan_power_timeseries(df_hourly, p_cen)
#     res_mag = plan_power_timeseries(df_hourly, p_mag)

#     state   = ATESState(cap_kWh_th=ates_cfg.storage_cap_kWh_th,
#                         monthly_loss_pct=ates_cfg.standing_loss_pct_per_month)
#     res_ates_mag = plan_power_timeseries_with_ates(df_hourly, p_mag, ates_cfg, state)

#     # KPI extraction (guards inside)
#     k_cen = kpis_from_results(res_cen)
#     k_mag = kpis_from_results(res_mag)
#     k_amg = kpis_from_results(res_ates_mag)

#     # Annual energy cost (flat)
#     cost_cen = k_cen["kWh"] * price
#     cost_mag = k_mag["kWh"] * price
#     cost_amg = k_amg["kWh"] * price

#     # CapEx — chiller (peak total cooling), piping common, ATES stack (wells+pumps+HX)
#     Qsize_kW = float(pd.concat([res_cen['Qsum_kW'], res_mag['Qsum_kW']]).max())
#     cap_cen = chiller_capex_usd_from_peak_Q(Qsize_kW)
#     cap_mag = chiller_capex_usd_from_peak_Q(Qsize_kW)

#     cap_piping = piping_capex_usd()
#     cap_baseline_other = BASELINE_FIXED_INSTALL_USD

#     cap_ates_wells = ates_wellfield_capex_usd(ates_cfg)
#     cap_ates_pumps = ates_pump_capex_usd(ates_cfg)
#     cap_ates_hx    = ates_hx_capex_usd(duty_kW=ates_cfg.qmax_discharge_kW, cfg=ates_cfg)

#     capex_cen = cap_cen + cap_piping + cap_baseline_other
#     capex_mag = cap_mag + cap_piping + cap_baseline_other
#     capex_amg = cap_mag + cap_piping + cap_ates_wells + cap_ates_pumps + cap_ates_hx

#     # O&M parity (same % of CapEx)
#     opex_cen = BASELINE_OPEX_PCT_PER_YEAR * capex_cen
#     opex_mag = BASELINE_OPEX_PCT_PER_YEAR * capex_mag
#     opex_amg = BASELINE_OPEX_PCT_PER_YEAR * capex_amg

#     # Tables
#     tbl = pd.DataFrame({
#         "Plan": ["Centrifugal", "Magnetic", "ATES+Magnetic"],
#         "kWh/yr": [k_cen["kWh"], k_mag["kWh"], k_amg["kWh"]],
#         "Cost $/yr": [cost_cen, cost_mag, cost_amg],
#         "Peak kW": [k_cen["peak_kW"], k_mag["peak_kW"], k_amg["peak_kW"]],
#         "PUE′ avg": [k_cen["PUEp_avg"], k_mag["PUEp_avg"], k_amg["PUEp_avg"]],
#         "PUE′ p95": [k_cen["PUEp_p95"], k_mag["PUEp_p95"], k_amg["PUEp_p95"]],
#         "CapEx $": [capex_cen, capex_mag, capex_amg],
#         "O&M $/yr": [opex_cen, opex_mag, opex_amg],
#     })

#     # Deltas vs Centrifugal
#     d_cen_mag_kWh  = k_cen["kWh"] - k_mag["kWh"]
#     d_cen_mag_cost = cost_cen - cost_mag
#     d_cen_amg_kWh  = k_cen["kWh"] - k_amg["kWh"]
#     d_cen_amg_cost = cost_cen - cost_amg

#     deltas = pd.DataFrame({
#         "Comparison": ["Cen→Mag", "Cen→ATES+Mag"],
#         "ΔkWh/yr": [d_cen_mag_kWh, d_cen_amg_kWh],
#         "ΔCost $/yr": [d_cen_mag_cost, d_cen_amg_cost],
#         "ΔPeak kW": [k_cen["peak_kW"] - k_mag["peak_kW"], k_cen["peak_kW"] - k_amg["peak_kW"]],
#     })

#     # Finance — incremental (NPV, IRR, simple payback)
#     premium_mag = capex_mag - capex_cen
#     npv_mag = npv_of_savings_usd(d_cen_mag_cost, LIFE_YEARS, DISCOUNT_RATE, ESCALATION)
#     irr_mag = irr_on_premium(premium_mag, d_cen_mag_cost, LIFE_YEARS, ESCALATION)
#     spb_mag = (premium_mag / d_cen_mag_cost) if d_cen_mag_cost > 0 else float('inf')

#     premium_amg = capex_amg - capex_cen
#     annual_sav_amg = (cost_cen + opex_cen) - (cost_amg + opex_amg)  # include O&M effect
#     npv_amg = npv_of_savings_usd(annual_sav_amg, LIFE_YEARS, DISCOUNT_RATE, ESCALATION)
#     irr_amg = irr_on_premium(premium_amg, annual_sav_amg, LIFE_YEARS, ESCALATION)
#     spb_amg = (premium_amg / annual_sav_amg) if annual_sav_amg > 0 else float('inf')

#     finance = pd.DataFrame({
#         "Case": ["Mag vs Cen", "ATES+Mag vs Cen"],
#         "Premium CapEx $": [premium_mag, premium_amg],
#         "NPV of savings $": [npv_mag, npv_amg],
#         "IRR": [irr_mag, irr_amg],
#         "Simple Payback (yrs)": [spb_mag, spb_amg],
#     })

#     # Sanity guards / warnings (no negatives allowed)
#     for col in ["kWh/yr", "Cost $/yr", "Peak kW", "CapEx $", "O&M $/yr"]:
#         if (np.array(tbl[col]) < 0).any():
#             print(f"WARNING: Negative values detected in {col}. Check inputs.")

#     print(f"\n=== {site_name} — Annual KPIs & CapEx ===")
#     print(tbl.to_string(index=False))
#     print("\nΔ vs Centrifugal:")
#     print(deltas.to_string(index=False))
#     print("\nFinance (incremental):")
#     print(finance.to_string(index=False))

# # Use your real 8760 frames per site
# # (these should already exist from your loader block)
# # df_phoenix, df_fairbanks

# # Fairbanks
# run_site_compare(df_fairbanks, "Fairbanks_Alaska")

# # Phoenix
# run_site_compare(df_phoenix, "Phoenix_Arizona")



In [6]:
# =============================================================================
# BLOCK 1 — INPUTS: read your 8760s + set site-specific knobs
# =============================================================================

import pandas as pd
import numpy as np
from dataclasses import dataclass

# --- 1A) Paths to your Excel files (use r-strings for Windows backslashes) ---
# Phoenix (hot) & Fairbanks (cold)
PHOENIX_XLSX   = r"C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx"
FAIRBANKS_XLSX = r"C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx"

# --- 1B) How we map columns from your files to the engine expected columns ----
# Your files have: ["Date/Time", "Qserver_kW", "Twb_in_C", "Tdb_out_C"]
# The engine wants for CRAH/WSE/ATES:  ["Qserver_kW", "Twb_out_C"]  (and we also keep Tdb_out_C for CRAC)
#
# IMPORTANT: You supplied *indoor* wet-bulb (Twb_in_C ~ 12–14 °C), not *outdoor* wet-bulb.
# For now (to keep moving), we approximate outdoor WB with dry-bulb for dispatch:
#   Twb_out_C ≈ Tdb_out_C   (this is conservative in hot/dry climates and slightly optimistic in humid)
# When you obtain RH (or WB), we will replace this with true Twb_out_C.
#
def load_site_frame(xlsx_path: str) -> pd.DataFrame:
    # If your data are in the first sheet and columns are as shown, this will work.
    # Otherwise, add: sheet_name="...", skiprows=..., usecols=...
    df = pd.read_excel(xlsx_path)

    # Standardize column names (strip spaces etc.)
    df.columns = [c.strip().replace(" ", "") for c in df.columns]
    # Expecting: "Date/Time"→"Date/Time" (or "Date/Time"), "Qserver_kW", "Twb_in_C", "Tdb_out_C"
    # Keep a datetime index if you like; it isn't required for the engine.
    if "Date/Time" in df.columns:
        # Make a best-effort datetime using a dummy year if needed
        df["DateTime"] = pd.to_datetime(df["Date/Time"], errors="coerce", format="mixed")
    else:
        df["DateTime"] = pd.NaT

    # Build the engine-ready frame
    out = pd.DataFrame({
        "DateTime": df["DateTime"],
        "Qserver_kW": pd.to_numeric(df["Qserver_kW"], errors="coerce"),
        # Approximate outdoor wet-bulb as dry-bulb for now (see note above)
        "Twb_out_C": pd.to_numeric(df["Tdb_out_C"], errors="coerce"),
        "Tdb_out_C": pd.to_numeric(df["Tdb_out_C"], errors="coerce"),
        # Keep Twb_in_C around if you later want CRAC-only checks
        "Twb_in_C": pd.to_numeric(df.get("Twb_in_C", np.nan), errors="coerce"),
    }).dropna(subset=["Qserver_kW", "Twb_out_C"])
    return out

df_phoenix   = load_site_frame(PHOENIX_XLSX)
df_fairbanks = load_site_frame(FAIRBANKS_XLSX)

print("Loaded source — Phoenix:", PHOENIX_XLSX)
print("Loaded source — Fairbanks:", FAIRBANKS_XLSX)
print("Loaded rows: Phoenix =", len(df_phoenix), "Fairbanks =", len(df_fairbanks))
print("Qserver first hour — Phoenix:", float(df_phoenix.loc[0, "Qserver_kW"]), "Fairbanks:", float(df_fairbanks.loc[0, "Qserver_kW"]))

# --- 1C) Tariffs (flat) + finance knobs (hard-coded per your choices) ---
SITES = {
    "Phoenix":   {"energy_price_usd_per_kWh": 0.10},  # your updated 10¢/kWh
    "Fairbanks": {"energy_price_usd_per_kWh": 0.08},  # example; edit if you have a local rate
}

DISCOUNT_RATE = 0.05         # 5% real
ESCALATION    = 0.02         # 2% real
LIFE_YEARS    = 20

# --- 1D) Chiller CapEx $/ton using Table 7 ratio (paper: magnetic 6.40M vs centrifugal 4.88294M) ---
# Ratio ≈ 6.40 / 4.88294 ≈ 1.311 → magnetic ≈ +31.1% premium
CHILLER_CAPEX_PER_TON_CEN_USD = 800.0
CHILLER_CAPEX_PER_TON_MAG_USD = CHILLER_CAPEX_PER_TON_CEN_USD * 1.311  # ≈ 1,049 $/ton
# (If you later compute absolute $ from that paper directly, swap these with site-size-specific totals.)

# --- 1E) Common piping & baseline "other" (parity placeholders) ---
PIPING_TOTAL_LENGTH_M = 500.0     # applied equally to all plans for comparison
PIPING_COST_PER_M_USD = 3000.0
BASELINE_FIXED_INSTALL_USD = 150_000.0
BASELINE_OPEX_PCT_PER_YEAR = 0.05

# --- 1F) ATES configs (dispatch levers set per your ask) ------------------
@dataclass
class ATESConfig:
    cold_well_C: float
    hot_well_C:  float
    hx_approach_K: float = 1.5     # tighten approach to help ATES
    storage_cap_kWh_th: float = 400_000.0  # ↑ capacity so it can span many hot hours
    standing_loss_pct_per_month: float = 0.02

    # Dispatch: FIRST RUN → disable charging to isolate discharge benefit
    qmax_discharge_kW: float = 5000.0       # ↑ discharge cap so ATES can actually carry load
    qmax_charge_kW: float = 0.0             # disable at first; re-enable later with threshold
    charge_when_twb_below_C: float = 10.0   # when re-enabled, WB must be cool (edit per site)

    # Wells & drilling
    n_cold_wells: int = 5
    n_warm_wells: int = 5
    well_depth_m: float = 80.0
    drilling_cost_per_m_usd: float = 300.0
    per_well_fixed_adder_usd: float = 1892.0
    casing_screen_adder_usd: float = 0.0

    # Hydraulics
    total_flow_m3_per_h: float = 1000.0
    head_m: float = 25.0                  # ↓ from 40 → 25 m to lower pump kWh (design lever)
    pump_hydraulic_eff: float = 0.78
    motor_eff: float = 0.96
    vfd_eff: float = 0.98

    # ATES HX CapEx proxy (placeholder)
    hx_capex_a_usd_per_kW: float = 50.0
    hx_capex_b_usd: float = 20_000.0

ATES_BY_SITE = {
    # You asked not to “change everything” for Austin earlier; now we’re using Phoenix/Fairbanks:
    "Phoenix":   ATESConfig(cold_well_C=6.0, hot_well_C=15.0, charge_when_twb_below_C=16.0),
    "Fairbanks": ATESConfig(cold_well_C=3.0, hot_well_C=12.0, charge_when_twb_below_C=8.0),
}


Loaded source — Phoenix: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\PHOENIX_DataCenter_Profile.xlsx
Loaded source — Fairbanks: C:\Users\apurv\OneDrive\Desktop\THESIS\My Notes\Data Files\FAIRBANKS_DataCenter_Profile.xlsx
Loaded rows: Phoenix = 8760 Fairbanks = 8760
Qserver first hour — Phoenix: 1342.595476084541 Fairbanks: 1342.595476084541


In [7]:
# =============================================================================
# BLOCK 2 — FUNCTIONS / CALCULATIONS
#  - Uses your engine's PlantParams, plan_power_timeseries
#  - ATES wrapper with: charge OFF initially, discharge before chiller
#  - CapEx uses separate $/ton for cen vs mag (Table 7 ratio)
# =============================================================================

# SAFETY: all helpers clamp to non-negative to prevent negative powers/flows.

class ATESState:
    def __init__(self, init_kWh_th=None, cap_kWh_th=400_000.0, monthly_loss_pct=0.02):
        self.cap_kWh_th = float(cap_kWh_th)
        self.E_kWh_th   = float(init_kWh_th) if init_kWh_th is not None else 0.5 * self.cap_kWh_th
        self.loss_per_hour = 1.0 - (1.0 - monthly_loss_pct)**(1.0/(30*24))  # convert %/mo → hourly

def clamp(x): return float(x) if x > 0 else 0.0

def pump_power_kW(flow_m3h, head_m, eta_h, eta_m, eta_vfd):
    rho=1000.0; g=9.81
    Q  = max(0.0, flow_m3h)/3600.0
    H  = max(0.0, head_m)
    eta_tot = max(1e-3, (eta_h or 0.0)*(eta_m or 0.0)*(eta_vfd or 0.0))
    return (rho*g*Q*H/1000.0)/eta_tot

def piping_capex_usd():
    return max(0.0, PIPING_TOTAL_LENGTH_M)*max(0.0, PIPING_COST_PER_M_USD)

def chiller_capex_usd_from_peak_Q(peak_Q_kW: float, per_ton_usd: float) -> float:
    tons = max(0.0, peak_Q_kW)/3.517
    return tons*max(0.0, per_ton_usd)

def ates_wellfield_capex_usd(cfg: ATESConfig) -> float:
    n = max(0, cfg.n_cold_wells + cfg.n_warm_wells)
    drill = max(0.0, cfg.well_depth_m)*max(0.0, cfg.drilling_cost_per_m_usd)*n
    add   = n*(max(0.0, cfg.per_well_fixed_adder_usd)+max(0.0, cfg.casing_screen_adder_usd))
    return drill + add

def ates_pump_capex_usd(cfg: ATESConfig) -> float:
    P = pump_power_kW(cfg.total_flow_m3_per_h, cfg.head_m, cfg.pump_hydraulic_eff, cfg.motor_eff, cfg.vfd_eff)
    return P*350.0  # your $/kW

def ates_hx_capex_usd(duty_kW: float, cfg: ATESConfig) -> float:
    return max(0.0, cfg.hx_capex_a_usd_per_kW)*max(0.0, duty_kW) + max(0.0, cfg.hx_capex_b_usd)

def npv_of_savings_usd(S, years=20, d=0.05, esc=0.02):
    S = max(0.0, float(S))
    tot=0.0
    for t in range(1, years+1):
        tot += S*((1+esc)**(t-1))/((1+d)**t)
    return tot

def irr_on_premium(premium_usd, annual_sav_usd, years=20, esc=0.02):
    prem=float(premium_usd); S=float(annual_sav_usd)
    cash = [-prem]+[S*((1+esc)**t) for t in range(1, years+1)]
    lo,hi=-0.9,1.5
    for _ in range(200):
        mid=0.5*(lo+hi)
        npv=sum(cash[t]/((1+mid)**t) for t in range(len(cash)))
        if npv>0: lo=mid
        else: hi=mid
    return 0.5*(lo+hi)

def kpis_from_results(res_df: pd.DataFrame):
    qsum = np.maximum(1e-9, res_df["Qsum_kW"].to_numpy(dtype=float))
    it = qsum / 1.10
    P  = np.maximum(0.0, res_df["P_sys_kW"].to_numpy(dtype=float))
    kWh=float(P.sum())       # sum of hourly kW = kWh/year (DO NOT multiply by 8760 again)
    peak_psys=float(P.max())
    peak_it=float(it.max())
    peak_qsum=float(qsum.max())
    puep=1.0+(P/it)
    return {"kWh":kWh, "peak_kW":peak_psys, "peak_Psys_kW":peak_psys,
            "peak_IT_kW":peak_it, "peak_Qsum_kW":peak_qsum,
            "PUEp_avg":float(np.mean(puep)), "PUEp_p95":float(np.quantile(puep,0.95))}

def plan_power_timeseries_with_ates(df_hourly: pd.DataFrame, plant: PlantParams, cfg: ATESConfig, state: ATESState, dt=1.0):
    """Insert ATES BEFORE the chiller. Charging is OFF initially (qmax_charge_kW=0)."""
    base = plan_power_timeseries(df_hourly.copy(), plant)

    # Feasibility: can ATES meet supply temperature?
    # Full-free if ATES can meet supply; partial-assist if it can cool return    
    Tcw_supply = getattr(plant, "Tcw_supply_set_C", 6.0)
    # If your engine exposes return, use it; else assume ΔT=6 K
    Tcw_return = getattr(plant, "Tcw_return_set_C", Tcw_supply + getattr(plant, "Tcw_deltaT_K", 6.0))

    #Tcw_return = getattr(plant, "Tcw_return_set_C", Tcw_supply + 6.0)

    can_full    = (cfg.cold_well_C + cfg.hx_approach_K) <= Tcw_supply
    can_partial = (cfg.cold_well_C + cfg.hx_approach_K) <= Tcw_return


    # Constant well-pump power when ATES is flowing (you can refine to variable-flow later)
    P_ates_pump = pump_power_kW(cfg.total_flow_m3_per_h, cfg.head_m, cfg.pump_hydraulic_eff, cfg.motor_eff, cfg.vfd_eff)

    base["Q_ates_kW"]=0.0; base["P_ates_pumps_kW"]=0.0; base["mode_with_ates"]=base["mode"].values

    for i in range(len(base)):
        Qsum = clamp(base.at[i,"Qsum_kW"])
        Qwse = clamp(base.at[i,"Q_wse_kW"])
        need = max(0.0, Qsum - Qwse)  # load remaining after WSE

        # --- Discharge ONLY when we would otherwise run the chiller (need > 0) ---
        Q_ates=0.0
        if need>1e-9 and can_partial and state.E_kWh_th>1e-9:
            E_avail_kW = state.E_kWh_th/max(1e-9, dt)
            Q_ates = min(need, cfg.qmax_discharge_kW, E_avail_kW)
            Q_ates = max(0.0, Q_ates)
            state.E_kWh_th = max(0.0, state.E_kWh_th - Q_ates*dt)
            base.at[i,"Q_ates_kW"] = Q_ates
            base.at[i,"P_ates_pumps_kW"] += P_ates_pump
            if can_full and abs(Q_ates-need)<1e-6:
                base.at[i,"mode_with_ates"] = "MODE1_FULL_FREE"
            else:
                base.at[i,"mode_with_ates"] = "MODE2_PARTIAL_FREE_ATES"


        # --- Charging (OFF for first pass): qmax_charge_kW == 0 keeps this inert ---
        # If you re-enable, gate with outdoor wet-bulb:
        Twb = float(df_hourly.iloc[i]["Twb_out_C"])
        if (cfg.qmax_charge_kW>1e-9) and (state.E_kWh_th<state.cap_kWh_th) and (Twb<=cfg.charge_when_twb_below_C):
            Qc = min(cfg.qmax_charge_kW, (state.cap_kWh_th-state.E_kWh_th)/max(1e-9,dt))
            if Qc>1e-9:
                base.at[i,"P_ates_pumps_kW"] += P_ates_pump
                state.E_kWh_th = min(state.cap_kWh_th, state.E_kWh_th + Qc*dt)

        # Standing loss
        state.E_kWh_th = max(0.0, state.E_kWh_th*(1.0 - state.loss_per_hour))

        # Recompute chiller after ATES
        Q_ch_old = clamp(base.at[i,"Q_chiller_kW"])
        Q_ch_new = max(0.0, Q_ch_old - Q_ates)
        base.at[i,"Q_chiller_kW"]=Q_ch_new
        COP = float(base.at[i,"COP"]) if (base.at[i,"COP"]==base.at[i,"COP"]) else 0.0
        P_ch_new = (Q_ch_new/max(1e-9,COP)) if (Q_ch_new>0 and COP>0) else 0.0
        base.at[i,"P_chiller_kW"]=P_ch_new

        # Sum all plant power + ATES pumps
        P_sys = (P_ch_new
                 + clamp(base.at[i,"P_pumps_chw_kW"])
                 + clamp(base.at[i,"P_pumps_ctw_kW"])
                 + clamp(base.at[i,"P_tower_fans_kW"])
                 + clamp(base.at[i,"P_crah_kW"])
                 + clamp(base.at[i,"P_ates_pumps_kW"]))
        base.at[i,"P_sys_kW"]=P_sys

    return base


In [8]:
# =============================================================================
# BLOCK 3 — OUTPUTS: Phoenix (hot) and Fairbanks (cold)
# =============================================================================

def finance_compare(df_hourly: pd.DataFrame, site_name: str):
    price = SITES[site_name]["energy_price_usd_per_kWh"]
    cfg   = ATES_BY_SITE[site_name]

    # Engine plants
    p_cen = PlantParams(chiller_type="centrifugal", chiller_cap_kW_per_unit=4058.0)
    p_mag = PlantParams(chiller_type="magnetic",    chiller_cap_kW_per_unit=3900.0)

    # Physics
    res_cen = plan_power_timeseries(df_hourly, p_cen)
    res_mag = plan_power_timeseries(df_hourly, p_mag)
    state   = ATESState(cap_kWh_th=cfg.storage_cap_kWh_th, monthly_loss_pct=cfg.standing_loss_pct_per_month)
    res_amg = plan_power_timeseries_with_ates(df_hourly, p_mag, cfg, state)

    # KPIs
    k_cen, k_mag, k_amg = map(kpis_from_results, (res_cen, res_mag, res_amg))
    cost_cen, cost_mag, cost_amg = k_cen["kWh"]*price, k_mag["kWh"]*price, k_amg["kWh"]*price

    # CapEx (now with different $/ton by chiller type)
    Qsize_kW = float(pd.concat([res_cen["Qsum_kW"], res_mag["Qsum_kW"]]).max())
    cap_cen = chiller_capex_usd_from_peak_Q(Qsize_kW, CHILLER_CAPEX_PER_TON_CEN_USD)
    cap_mag = chiller_capex_usd_from_peak_Q(Qsize_kW, CHILLER_CAPEX_PER_TON_MAG_USD)

    cap_piping = piping_capex_usd()
    cap_base_other = BASELINE_FIXED_INSTALL_USD

    # Component O&M rates (closer to practice than flat 5%)
    ATES_OPEX_RATES = {
        "wells": 0.02,   # 2%/yr
        "pumps": 0.04,   # 4%/yr
        "hx":    0.03,   # 3%/yr
        "piping":0.01,   # 1%/yr
    }
    
    # Compute ATES component CapEx
    cap_ates_wells = ates_wellfield_capex_usd(cfg)
    cap_ates_pumps = ates_pump_capex_usd(cfg)
    cap_ates_hx    = ates_hx_capex_usd(cfg.qmax_discharge_kW, cfg)
    cap_piping     = piping_capex_usd()
    
    # Plan CapEx (unchanged)
    capex_cen = cap_cen + cap_piping + BASELINE_FIXED_INSTALL_USD
    capex_mag = cap_mag + cap_piping + BASELINE_FIXED_INSTALL_USD
    capex_amg = cap_mag + cap_piping + cap_ates_wells + cap_ates_pumps + cap_ates_hx
    
    # O&M: baseline stays at your parity placeholder; ATES uses component rates
    opex_cen = BASELINE_OPEX_PCT_PER_YEAR * capex_cen
    opex_mag = BASELINE_OPEX_PCT_PER_YEAR * capex_mag
    opex_amg = (ATES_OPEX_RATES["wells"]  * cap_ates_wells +
                ATES_OPEX_RATES["pumps"]  * cap_ates_pumps +
                ATES_OPEX_RATES["hx"]     * cap_ates_hx    +
                ATES_OPEX_RATES["piping"] * cap_piping     +
                BASELINE_OPEX_PCT_PER_YEAR * cap_mag)  # keep chiller-side bits same as Mag


    tbl = pd.DataFrame({
        "Plan":["Centrifugal","Magnetic","ATES+Magnetic"],
        "kWh/yr":[k_cen["kWh"], k_mag["kWh"], k_amg["kWh"]],
        "Cost $/yr":[cost_cen, cost_mag, cost_amg],
        "Peak P_sys kW":[k_cen["peak_Psys_kW"], k_mag["peak_Psys_kW"], k_amg["peak_Psys_kW"]],
        "Peak IT kW":[k_cen["peak_IT_kW"], k_mag["peak_IT_kW"], k_amg["peak_IT_kW"]],
        "Peak Qsum kW":[k_cen["peak_Qsum_kW"], k_mag["peak_Qsum_kW"], k_amg["peak_Qsum_kW"]],
        "PUE′ avg":[k_cen["PUEp_avg"], k_mag["PUEp_avg"], k_amg["PUEp_avg"]],
        "PUE′ p95":[k_cen["PUEp_p95"], k_mag["PUEp_p95"], k_amg["PUEp_p95"]],
        "CapEx $":[capex_cen, capex_mag, capex_amg],
        "O&M $/yr":[opex_cen, opex_mag, opex_amg],
    })
    deltas = pd.DataFrame({
        "Comparison":["Cen→Mag","Cen→ATES+Mag"],
        "ΔkWh/yr":[k_cen["kWh"]-k_mag["kWh"], k_cen["kWh"]-k_amg["kWh"]],
        "ΔCost $/yr":[(k_cen["kWh"]-k_mag["kWh"])*price, (k_cen["kWh"]-k_amg["kWh"])*price],
        "ΔPeak P_sys kW":[k_cen["peak_Psys_kW"]-k_mag["peak_Psys_kW"], k_cen["peak_Psys_kW"]-k_amg["peak_Psys_kW"]],
    })

    # Finance deltas (incremental)
    premium_mag = capex_mag - capex_cen
    npv_mag = npv_of_savings_usd((k_cen["kWh"]-k_mag["kWh"])*price, LIFE_YEARS, DISCOUNT_RATE, ESCALATION)
    irr_mag = irr_on_premium(premium_mag, (k_cen["kWh"]-k_mag["kWh"])*price, LIFE_YEARS, ESCALATION)
    spb_mag = (premium_mag / ((k_cen["kWh"]-k_mag["kWh"])*price)) if (k_cen["kWh"]>k_mag["kWh"]) else float("inf")

    premium_amg = capex_amg - capex_cen
    annual_sav_amg = (k_cen["kWh"]-k_amg["kWh"])*price + (opex_cen - opex_amg)
    npv_amg = npv_of_savings_usd(annual_sav_amg, LIFE_YEARS, DISCOUNT_RATE, ESCALATION)
    irr_amg = irr_on_premium(premium_amg, annual_sav_amg, LIFE_YEARS, ESCALATION)
    spb_amg = (premium_amg / annual_sav_amg) if (annual_sav_amg>0) else float("inf")

    finance = pd.DataFrame({
        "Case":["Mag vs Cen","ATES+Mag vs Cen"],
        "Premium CapEx $":[premium_mag, premium_amg],
        "NPV of savings $":[npv_mag, npv_amg],
        "IRR":[irr_mag, irr_amg],
        "Simple Payback (yrs)":[spb_mag, spb_amg],
    })

    print(f"\n=== {site_name} — Annual KPIs & CapEx ===")
    print(tbl.to_string(index=False))
    print("\nΔ vs Centrifugal:")
    print(deltas.to_string(index=False))
    print("\nFinance (incremental):")
    print(finance.to_string(index=False))

    # Publish fresh per-site outputs for chapter4_plots.ipynb.
    # This prevents the plotting notebook from falling back to stale globals.
    suffix = "phx" if site_name == "Phoenix" else "fb"
    g = globals()
    g[f"res_cen_{suffix}"] = res_cen
    g[f"res_mag_{suffix}"] = res_mag
    g[f"res_amg_{suffix}"] = res_amg
    g[f"kpi_table_{suffix}"] = tbl
    g[f"deltas_{suffix}"] = deltas
    g[f"finance_{suffix}"] = finance
    g[f"capex_cen_{suffix}"] = capex_cen
    g[f"capex_mag_{suffix}"] = capex_mag
    g[f"capex_amg_{suffix}"] = capex_amg
    g[f"opex_cen_{suffix}"] = opex_cen
    g[f"opex_mag_{suffix}"] = opex_mag
    g[f"opex_amg_{suffix}"] = opex_amg
    g[f"spb_mag_{suffix}"] = spb_mag
    g[f"spb_amg_{suffix}"] = spb_amg

    return {"kpis": tbl, "deltas": deltas, "finance": finance,
            "res_cen": res_cen, "res_mag": res_mag, "res_amg": res_amg}

# -------- RUN BOTH SITES --------
RESULTS = {}
RESULTS["Phoenix"] = finance_compare(df_phoenix,   "Phoenix")
RESULTS["Fairbanks"] = finance_compare(df_fairbanks, "Fairbanks")




=== Phoenix — Annual KPIs & CapEx ===
         Plan       kWh/yr     Cost $/yr  Peak P_sys kW  Peak IT kW  Peak Qsum kW  PUE′ avg  PUE′ p95      CapEx $      O&M $/yr
  Centrifugal 2.294010e+06 229400.950136     463.533033 1587.418507   1746.160358  1.185724  1.313283 2.047193e+06 102359.657182
     Magnetic 1.706964e+06 170696.394975     436.466085 1587.418507   1746.160358  1.138086  1.252713 2.170720e+06 108536.010565
ATES+Magnetic 1.704549e+06 170454.899987     436.466085 1587.418507   1746.160358  1.137900  1.252713 2.582133e+06  55614.109131

Δ vs Centrifugal:
  Comparison       ΔkWh/yr   ΔCost $/yr  ΔPeak P_sys kW
     Cen→Mag 587045.551614 58704.555161       27.066948
Cen→ATES+Mag 589460.501491 58946.050149       27.066948

Finance (incremental):
           Case  Premium CapEx $  NPV of savings $      IRR  Simple Payback (yrs)
     Mag vs Cen    123527.067670      8.609259e+05 0.504537              2.104216
ATES+Mag vs Cen    534939.531803      1.550010e+06 0.215484           